# Person detection in drone video (VisDrone) — tuned for COCO **AP-small**

Detection only (no tracking). Frames are processed independently, so there are no IDs and no temporal state.

**What this notebook combines**

| Idea | Source | Where |
|---|---|---|
| P2 (stride-4) head | both | `yolov8?-p2.yaml` in the training cell |
| Tiled training **and** tiled (SAHI-style) inference, same tile size | both (+ fix of a train/test mismatch) | tiling + `SlicedDetector` |
| Up-scaling tiles (`IMGSZ/TILE` = 1.5x) instead of down-scaling whole frames | mine | config |
| NWD + CIoU box loss | both | patch cell |
| Motion-blur / defocus / noise / exposure / JPEG augmentation | both | patch cell |
| Merge *pedestrian*+*people* -> `person`; ignored regions painted grey for training and discarded at evaluation | both | records + eval |
| Box-level copy-paste of small persons (built-in `copy_paste` needs masks, VisDrone has boxes only) | both (+ fix) | tiling |
| Real video frames (VisDrone-VID) in training for real motion blur | mine | `USE_VID_TRAIN` |
| Blur-aware confidence threshold for video output | new | video cell |
| COCO evaluation (AP_small, by blur level, sliced vs full-frame) | mine | evaluation cells |

**Google Drive:** the dataset is read from `My Drive/visdrone_data/` (you upload it once, nothing is downloaded), and everything produced is written straight to `My Drive/visdrone_person/` (`runs/` training runs + weights saved every epoch, `results/` metrics, predictions, demo video, `weights/best.pt`).

**How to run:** Runtime > Change runtime type > **T4 GPU**, then *Run all*. Set `QUICK_TEST = True` in the settings cell first if you only want to check that everything runs (~20-30 min).
Training time depends on `MAX_TRAIN_TILES` x `EPOCHS`; the defaults are sized for a few hours on a T4. If Colab disconnects, re-run the cells up to and including the patch cell and resume with the last cell.

## 1. Install

In [1]:
!pip -q install "ultralytics>=8.3,<8.4" pycocotools

## 2. Settings

In [2]:
import os, sys, json, math, random, shutil, subprocess, time, glob, io, contextlib
from pathlib import Path
from collections import defaultdict
import numpy as np
import cv2
from PIL import Image
from tqdm.auto import tqdm
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")   # less GPU-memory fragmentation (must be set before torch starts)

# =====================  USER SETTINGS  =====================
QUICK_TEST      = False   # True = ~20-30 min pipeline check (tiny model, tiny data). Set False for the real run.
MODEL_SIZE      = "s"     # "n" | "s" | "m"  (s fits a free T4; m is better but ~2x slower)
TILE            = 512     # native-resolution crop size used for training AND inference (must match!)
OVERLAP         = 0.25    # tile overlap (must exceed typical person size so no person is cut in all tiles)
IMGSZ           = 768     # network input size. IMGSZ/TILE = 1.5x upscaling -> tiny people get more pixels
EPOCHS          = 30
BATCH           = 8       # 16 ran out of GPU memory in the box-matching step on a T4 (crowded tiles); 8 is safer
MAX_TRAIN_TILES = 16000   # cap on training tiles (time budget). None = use all
EMPTY_TILE_FRAC = 0.10    # share of person-free tiles kept as hard negatives
COPY_PASTE_PROB = 0.30    # prob. a train tile receives pasted small persons (box-level copy-paste)
NWD_WEIGHT      = 0.5     # box loss = (1-w)*CIoU + w*NWD
NWD_C           = 12.8    # NWD normalising constant in network-input pixels (value from the NWD paper)
USE_VID_TRAIN   = True    # add frames from VisDrone-VID train (real video motion blur); ~7.5 GB
VID_TRAIN_STRIDE = 8      # keep every 8th video frame (neighbouring frames are near-duplicates)
VID_EVAL_STRIDE  = 10     # evaluate on every 10th VID-val frame
EVAL_MAX_IMAGES  = None   # limit images for evaluation (None = all)
WORKERS          = 2      # data-loader processes; use 1 if Colab runs out of RAM (each worker holds ~1-2 GB)
REBUILD_TILES    = False  # False = reuse tiles already on local disk (e.g. after a kernel restart); True = rebuild from scratch

# ---- Google Drive (nothing is downloaded: the dataset is READ from your Drive) ----
DRIVE_FOLDER_NAME = "visdrone_person_1"   # results are written to My Drive/<this folder>/
DATA_FOLDER_NAME  = "visdrone_data"     # YOU upload the VisDrone zips into My Drive/<this folder>/ (see section 3)
# ===========================================================

if QUICK_TEST:
    MODEL_SIZE, EPOCHS, BATCH, MAX_TRAIN_TILES = "n", 2, 8, 600
    USE_VID_TRAIN, EVAL_MAX_IMAGES, VID_EVAL_STRIDE = False, 40, 60

WORK = Path(os.environ.get("WORK", "/content/visdrone_work"))   # fast local disk for tiles / unzipped data
WORK.mkdir(parents=True, exist_ok=True)

# ---- mount Google Drive (required: results are written straight to it) ----
try:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_BASE = Path("/content/drive/MyDrive")
except ImportError:                                    # not on Colab (local testing only)
    DRIVE_BASE = WORK / "fake_drive"
    print("!! google.colab not found -> using", DRIVE_BASE, "instead of Google Drive")
OUT     = DRIVE_BASE / DRIVE_FOLDER_NAME
RUNS    = OUT / "runs"            # Ultralytics training runs (weights, curves) - written every epoch
RESULTS = OUT / "results"         # metrics, predictions, demo video, detections JSON
WEIGHTS = OUT / "weights"         # final best.pt copy
DATA_DIR = DRIVE_BASE / DATA_FOLDER_NAME   # input: the VisDrone zips (or unzipped folders) you uploaded
for d in (RUNS, RESULTS, WEIGHTS, DATA_DIR):
    d.mkdir(parents=True, exist_ok=True)

import gc
def ram(tag=""):
    """Print system RAM use. Call it anywhere to see where memory goes (Colab free = ~12.7 GB)."""
    try:
        import psutil
        vm, rss = psutil.virtual_memory(), psutil.Process().memory_info().rss
        kids = sum(p.memory_info().rss for p in psutil.Process().children(recursive=True))
        print(f"[RAM {tag}] system {vm.used/2**30:.1f}/{vm.total/2**30:.1f} GB | this notebook {rss/2**30:.1f} GB + workers {kids/2**30:.1f} GB")
    except Exception:
        pass

SEED = 0
random.seed(SEED); np.random.seed(SEED)
(RESULTS / "config.json").write_text(json.dumps(
    {k: v for k, v in globals().items() if k.isupper() and isinstance(v, (int, float, str, bool, type(None)))}, indent=1))
print("local work dir:", WORK, "\nGoogle Drive output:", OUT)
try:
    import torch
    print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE (enable Runtime > Change runtime type > T4 GPU)")
except ImportError:
    pass

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
local work dir: /content/visdrone_work 
Google Drive output: /content/drive/MyDrive/visdrone_person_1
GPU: Tesla T4


## 3. Load the dataset from your Google Drive (no downloading)
Upload the VisDrone files **once, by hand,** into `My Drive/visdrone_data/` and keep the original names. This cell only reads them from Drive:

| File to upload | Size | Needed for |
|---|---|---|
| `VisDrone2019-DET-train.zip` | 1.44 GB | training (**required**) |
| `VisDrone2019-DET-val.zip` | 0.07 GB | validation + AP_small on images (**required**) |
| `VisDrone2019-VID-val.zip` | 1.49 GB | AP_small on video frames + demo video (recommended) |
| `VisDrone2019-VID-train.zip` | 7.53 GB | real motion-blur frames for training (optional) |

Upload the zips as they are, do not unzip them in Drive (one big file copies far faster than thousands of images). An unzipped folder with the same name also works. Get the files from the VisDrone GitHub page (github.com/VisDrone/VisDrone-Dataset, Task 1 = DET, Task 2 = VID). Missing optional files are skipped with a message; missing required files stop with a clear error.

In [5]:
import os

f = "/content/drive/MyDrive/visdrone_data/VisDrone2019-DET-train.zip"

print("exists:", os.path.exists(f))
print("size:", os.path.getsize(f))

exists: True
size: 1549875511


In [9]:
for fn in [
    "VisDrone2019-DET-val.zip",
    "VisDrone2019-VID-val.zip",
]:
    path = f"/content/drive/MyDrive/visdrone_data/{fn}"
    try:
        with open(path, "rb") as f:
            print(fn, "OK", f.read(16))
    except Exception as e:
        print(fn, "FAIL", e)

VisDrone2019-DET-val.zip OK b'PK\x03\x04\n\x00\x00\x00\x00\x00\x1b~\x98L\x00\x00'
VisDrone2019-VID-val.zip FAIL [Errno 5] Input/output error


In [4]:
# Reads the dataset from YOUR Google Drive folder (DATA_DIR). Nothing is downloaded from the internet.
RAW = WORK / "raw"; RAW.mkdir(exist_ok=True)

SPLITS = {   # key -> name prefix of the zip / folder you uploaded
    "DET-train": "VisDrone2019-DET-train",
    "DET-val":   "VisDrone2019-DET-val",
    "VID-train": "VisDrone2019-VID-train",
    "VID-val":   "VisDrone2019-VID-val",
}

def _copy_low_ram(src, dst, chunk=64 * 2**20):
    """Copy a big Drive file to local disk in 64 MB chunks, dropping the OS file cache as we go
    (reading a multi-GB zip straight from the Drive mount can fill Colab's RAM)."""
    try:
        with open(src, "rb") as fi, open(dst, "wb") as fo:
            while True:
                b = fi.read(chunk)
                if not b:
                    break
                fo.write(b)
                for f in (fi, fo):
                    try:
                        f.flush(); os.fsync(f.fileno()) if f is fo else None
                        os.posix_fadvise(f.fileno(), 0, 0, os.POSIX_FADV_DONTNEED)
                    except Exception:
                        pass
    except OSError as e:
        if e.errno == 5: # Input/output error
            print("!! Low-RAM copy encountered a Drive mount I/O error. Retrying with standard shutil.copy...")
            time.sleep(3)
            try:
                shutil.copy2(src, dst)
            except OSError as e2:
                if e2.errno == 5:
                    print("!! Standard copy also failed with I/O error. Attempting to force-remount Google Drive...")
                    try:
                        from google.colab import drive
                        drive.mount("/content/drive", force_remount=True)
                        time.sleep(3)
                        print("!! Drive remounted. Trying copy one final time...")
                        shutil.copy2(src, dst)
                    except Exception as e3:
                        raise OSError(f"Failed to copy even after remounting Drive: {e3}") from e3
                else:
                    raise e2
        else:
            raise e

def _unzip(zpath, dest):
    subprocess.run(["unzip", "-q", "-o", str(zpath), "-d", str(dest)], check=True)

def _find_root(dest):
    a = next(Path(dest).rglob("annotations"), None)
    return a.parent if a else None

def fetch(name, required):
    """Copy/unzip one split from Drive to fast local disk. Returns the folder that holds 'annotations/', or None."""
    dest = RAW / name
    if dest.exists() and _find_root(dest):
        return _find_root(dest)
    prefix = SPLITS[name].lower()
    found = sorted(p for p in DATA_DIR.iterdir() if p.name.lower().startswith(prefix))
    try:
        if not found:
            raise FileNotFoundError(f"nothing named '{SPLITS[name]}*' in {DATA_DIR}")
        src = next((p for p in found if p.is_dir()), None) or next((p for p in found if p.suffix.lower() == ".zip"), None)
        if src is None:
            raise FileNotFoundError(f"'{found[0].name}' is neither a .zip nor a folder")
        dest.mkdir(parents=True, exist_ok=True)
        print(f"{name}: reading {src.name} from Drive ...")
        if src.is_dir():
            shutil.copytree(src, dest, dirs_exist_ok=True)
        else:
            local_zip = RAW / f"{name}.zip"
            _copy_low_ram(src, local_zip)      # Drive -> local disk first, then unzip from local disk
            _unzip(local_zip, dest)
            local_zip.unlink(missing_ok=True)
        root = _find_root(dest)
        assert root, f"no 'annotations' folder found inside {src.name} (is the zip complete?)"
        gc.collect(); ram(f"after {name}")
        return root
    except Exception as e:
        shutil.rmtree(dest, ignore_errors=True)
        msg = f"{name}: {e}"
        if required:
            have = [p.name for p in DATA_DIR.iterdir()] if DATA_DIR.exists() else []
            raise SystemExit(f"\n!! REQUIRED dataset missing -> {msg}\nFiles currently in {DATA_DIR}: {have}\n"
                             f"Upload the VisDrone zips there (see section 3 of this notebook), then re-run this cell.")
        print(f"(optional) {msg} -> skipping")
        return None

DET_TRAIN = fetch("DET-train", True)
DET_VAL   = fetch("DET-val", True)
VID_VAL   = fetch("VID-val", False)
VID_TRAIN = fetch("VID-train", False) if USE_VID_TRAIN else None
if USE_VID_TRAIN and VID_TRAIN is None:
    print("VID-train not found -> training on DET images only")
print("DET-train:", DET_TRAIN, "\nDET-val:  ", DET_VAL, "\nVID-train:", VID_TRAIN, "\nVID-val:  ", VID_VAL)

DET-train: reading VisDrone2019-DET-train.zip from Drive ...
!! Low-RAM copy encountered a Drive mount I/O error. Retrying with standard shutil.copy...
!! Standard copy also failed with I/O error. Attempting to force-remount Google Drive...
Mounted at /content/drive
!! Drive remounted. Trying copy one final time...


ERROR:root:Internal Python error in the inspect module.
Below is the traceback from this internal error.



Traceback (most recent call last):
  File "/tmp/ipykernel_5923/3462677420.py", line 17, in _copy_low_ram
    b = fi.read(chunk)
OSError: [Errno 5] Input/output error

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/tmp/ipykernel_5923/3462677420.py", line 32, in _copy_low_ram
    shutil.copy2(src, dst)
    ~~~~~~~~~~~~^^^^^^^^^^
  File "/usr/lib/python3.13/shutil.py", line 468, in copy2
    copyfile(src, dst, follow_symlinks=follow_symlinks)
    ~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.13/shutil.py", line 283, in copyfile
    copyfileobj(fsrc, fdst)
    ~~~~~~~~~~~^^^^^^^^^^^^
  File "/usr/lib/python3.13/shutil.py", line 203, in copyfileobj
    while buf := fsrc_read(length):
                 ~~~~~~~~~^^^^^^^^
OSError: [Errno 5] Input/output error

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/tmp/ipykernel_5923/346267742

TypeError: object of type 'NoneType' has no len()

## 4. Parse annotations
Categories 1 (pedestrian) and 2 (people) become one `person` class. Category-0 *ignored regions* are kept separately.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# VisDrone categories: 0 ignored-region, 1 pedestrian, 2 people, 3 bicycle, 4 car, ... -> we merge 1+2 into one class "person"
PERSON_CATS = (1, 2)

def _arr(x):
    return np.array(x, np.float32).reshape(-1, 4)

def parse_det_file(path):
    """DET row: x,y,w,h,score,category,truncation,occlusion  ->  (person boxes xyxy, ignore regions xyxy)"""
    boxes, ign = [], []
    p = Path(path)
    if p.exists():
        for line in p.read_text().splitlines():
            f = line.strip().split(",")
            if len(f) < 6:
                continue
            x, y, w, h, score, cat = (int(float(v)) for v in f[:6])
            if w <= 0 or h <= 0:
                continue
            b = (x, y, x + w, y + h)
            if cat == 0 or (score == 0 and cat in PERSON_CATS):
                ign.append(b)
            elif cat in PERSON_CATS:
                boxes.append(b)
    return _arr(boxes), _arr(ign)

def parse_vid_file(path):
    """VID row: frame,id,x,y,w,h,score,category,truncation,occlusion -> {frame: (boxes, ignore)}"""
    per = defaultdict(lambda: ([], []))
    for line in Path(path).read_text().splitlines():
        f = line.strip().split(",")
        if len(f) < 8:
            continue
        fr, _, x, y, w, h, score, cat = (int(float(v)) for v in f[:8])
        if w <= 0 or h <= 0:
            continue
        b = (x, y, x + w, y + h)
        if cat == 0 or (score == 0 and cat in PERSON_CATS):
            per[fr][1].append(b)
        elif cat in PERSON_CATS:
            per[fr][0].append(b)
    return {k: (_arr(v[0]), _arr(v[1])) for k, v in per.items()}

def det_records(root):
    root = Path(root); out = []
    for p in sorted((root / "images").glob("*.jpg")):
        b, i = parse_det_file(root / "annotations" / (p.stem + ".txt"))
        out.append(dict(uid=f"det_{p.stem}", path=str(p), boxes=b, ignore=i))
    return out

def vid_records(root, stride=1):
    root = Path(root); out = []
    for seq in sorted(d for d in (root / "sequences").iterdir() if d.is_dir()):
        ann = parse_vid_file(root / "annotations" / f"{seq.name}.txt")
        frames = sorted(seq.glob("*.jpg"))
        for p in frames[::stride]:
            b, i = ann.get(int(p.stem), (_arr([]), _arr([])))
            out.append(dict(uid=f"vid_{seq.name}_{p.stem}", path=str(p), boxes=b, ignore=i))
    return out

def summarize(records, name):
    n = sum(len(r["boxes"]) for r in records)
    sides = np.concatenate([np.sqrt((r["boxes"][:, 2]-r["boxes"][:, 0]) * (r["boxes"][:, 3]-r["boxes"][:, 1])) for r in records if len(r["boxes"])] or [np.zeros(0)])
    print(f"{name}: {len(records)} images, {n} person boxes, "
          f"{(sides < 32).mean()*100:.0f}% COCO-small (<32px side), median side {np.median(sides):.0f}px")

## 5. Tiling + copy-paste helpers

In [ ]:
def tile_origins(W, H, tile, overlap):
    step = max(1, int(tile * (1 - overlap)))
    def axis(L):
        if L <= tile:
            return [0]
        xs = list(range(0, L - tile + 1, step))
        if xs[-1] != L - tile:
            xs.append(L - tile)
        return xs
    return [(x, y) for y in axis(H) for x in axis(W)]

def boxes_in_tile(boxes, x0, y0, tw, th, min_vis=0.5):
    """Boxes (xyxy, image coords) -> boxes clipped to the tile; drops boxes <min_vis visible or <2px."""
    if len(boxes) == 0:
        return np.zeros((0, 4), np.float32)
    b = boxes.copy()
    b[:, [0, 2]] -= x0; b[:, [1, 3]] -= y0
    area0 = (b[:, 2] - b[:, 0]) * (b[:, 3] - b[:, 1])
    c = b.copy()
    c[:, [0, 2]] = c[:, [0, 2]].clip(0, tw); c[:, [1, 3]] = c[:, [1, 3]].clip(0, th)
    cw, ch = c[:, 2] - c[:, 0], c[:, 3] - c[:, 1]
    vis = (cw.clip(0) * ch.clip(0)) / np.maximum(area0, 1e-6)
    return c[(vis >= min_vis) & (cw >= 2) & (ch >= 2)]

def paint_ignore(img, ign):
    """Grey-out VisDrone 'ignored regions' so un-annotated people there are not learned as background."""
    for x1, y1, x2, y2 in ign.astype(int):
        img[max(y1, 0):max(y2, 0), max(x1, 0):max(x2, 0)] = 114

def plan_tiles(records, tile, overlap, min_vis=0.5):
    plan = []
    for ri, r in enumerate(tqdm(records, desc="planning tiles")):
        W, H = Image.open(r["path"]).size
        for x0, y0 in tile_origins(W, H, tile, overlap):
            tw, th = min(tile, W - x0), min(tile, H - y0)
            plan.append((ri, x0, y0, tw, th, len(boxes_in_tile(r["boxes"], x0, y0, tw, th, min_vis))))
    return plan

def sample_plan(plan, max_tiles, empty_frac, rng):
    pos = [p for p in plan if p[5] > 0]; emp = [p for p in plan if p[5] == 0]
    n_pos = len(pos) if max_tiles is None else min(len(pos), int(max_tiles * (1 - empty_frac)))
    n_emp = min(len(emp), int(n_pos * empty_frac / max(1 - empty_frac, 1e-6)))
    return rng.sample(pos, n_pos) + rng.sample(emp, n_emp)

# ---------- box-level copy-paste of SMALL persons (Ultralytics' copy_paste needs masks, VisDrone has only boxes) ----------
def _overlaps(box, others):
    """True if axis-aligned `box` (x1,y1,x2,y2) intersects any box in `others`."""
    if len(others) == 0:
        return False
    o = np.asarray(others, np.float32).reshape(-1, 4)
    ix = np.minimum(box[2], o[:, 2]) - np.maximum(box[0], o[:, 0])
    iy = np.minimum(box[3], o[:, 3]) - np.maximum(box[1], o[:, 1])
    return bool(((ix > 0) & (iy > 0)).any())

def add_donors(pool, crop, lab, rng, pool_size=3000, small_side=48, per_tile=2):
    th, tw = crop.shape[:2]; added = 0
    for i in rng.sample(range(len(lab)), len(lab)):
        x1, y1, x2, y2 = lab[i]; bw, bh = x2 - x1, y2 - y1
        if max(bw, bh) > small_side or added >= per_tile:
            continue
        m = max(3, int(0.3 * max(bw, bh)))
        reg = np.array([x1 - m, y1 - m, x2 + m, y2 + m])
        if reg[0] < 0 or reg[1] < 0 or reg[2] > tw or reg[3] > th:
            continue
        if _overlaps(reg, np.delete(lab, i, axis=0)):
            continue
        patch = crop[int(reg[1]):int(reg[3]), int(reg[0]):int(reg[2])].copy()
        if patch.size == 0:
            continue
        item = dict(patch=patch, m=m)
        if len(pool) < pool_size:
            pool.append(item)
        else:
            pool[rng.randrange(pool_size)] = item
        added += 1

def paste_donors(crop, lab, pool, rng, max_paste=5):
    th, tw = crop.shape[:2]; lab = lab.copy(); n = 0
    for _ in range(rng.randint(1, max_paste)):
        d = rng.choice(pool); patch, m = d["patch"], d["m"]; ph, pw = patch.shape[:2]
        if pw >= tw or ph >= th:
            continue
        for _try in range(10):
            px, py = rng.randint(0, tw - pw), rng.randint(0, th - ph)
            region = np.array([px, py, px + pw, py + ph], np.float32)
            if _overlaps(region, lab):
                continue
            alpha = np.zeros((ph, pw), np.float32)
            k = max(m // 2, 1)
            alpha[k:ph - k, k:pw - k] = 1.0
            ks = 2 * (m // 2) + 1
            alpha = cv2.GaussianBlur(alpha, (ks, ks), 0)[..., None]
            roi = crop[py:py + ph, px:px + pw].astype(np.float32)
            crop[py:py + ph, px:px + pw] = (alpha * patch + (1 - alpha) * roi).astype(np.uint8)
            lab = np.vstack([lab, [px + m, py + m, px + pw - m, py + ph - m]]).astype(np.float32)
            n += 1
            break
    return crop, lab, n

def write_tiles(records, selected, img_dir, lbl_dir, paint=True, copy_paste_prob=0.0, rng=None):
    rng = rng or random.Random(0)
    img_dir, lbl_dir = Path(img_dir), Path(lbl_dir)
    img_dir.mkdir(parents=True, exist_ok=True); lbl_dir.mkdir(parents=True, exist_ok=True)
    by_rec = defaultdict(list)
    for s in selected:
        by_rec[s[0]].append(s)
    order = list(by_rec); rng.shuffle(order)
    pool, n_tiles, n_boxes, n_pasted = [], 0, 0, 0
    for ri in tqdm(order, desc="writing tiles"):
        r = records[ri]
        img = cv2.imread(r["path"])
        if img is None:
            continue
        if paint:
            paint_ignore(img, r["ignore"])
        for _, x0, y0, tw, th, _n in by_rec[ri]:
            crop = img[y0:y0 + th, x0:x0 + tw].copy()
            lab = boxes_in_tile(r["boxes"], x0, y0, tw, th)
            if copy_paste_prob > 0:
                if len(lab):
                    add_donors(pool, crop, lab, rng)
                if pool and rng.random() < copy_paste_prob:
                    crop, lab, k = paste_donors(crop, lab, pool, rng); n_pasted += k
            name = f"{r['uid']}_{x0}_{y0}"
            cv2.imwrite(str(img_dir / f"{name}.jpg"), crop, [cv2.IMWRITE_JPEG_QUALITY, 95])
            with open(lbl_dir / f"{name}.txt", "w") as f:
                for x1, y1, x2, y2 in lab:
                    f.write(f"0 {(x1+x2)/2/tw:.6f} {(y1+y2)/2/th:.6f} {(x2-x1)/tw:.6f} {(y2-y1)/th:.6f}\n")
            n_tiles += 1; n_boxes += len(lab)
    print(f"wrote {n_tiles} tiles, {n_boxes} boxes ({n_pasted} pasted) -> {img_dir}")

## 6. Build the training / validation tile datasets

In [ ]:
rng = random.Random(SEED)
det_train = det_records(DET_TRAIN)
det_val   = det_records(DET_VAL)
vid_train = vid_records(VID_TRAIN, VID_TRAIN_STRIDE) if VID_TRAIN else []
vid_val   = vid_records(VID_VAL, VID_EVAL_STRIDE) if VID_VAL else []
for n, r in [("DET train", det_train), ("DET val", det_val), ("VID train (subsampled)", vid_train), ("VID val (subsampled)", vid_val)]:
    if r: summarize(r, n)

DS = WORK / "tiles"
DATA_YAML = DS / "data.yaml"      # written last, so its existence means the build finished
if DATA_YAML.exists() and not REBUILD_TILES:
    print("tiles already built -> skipping (set REBUILD_TILES = True to rebuild)")
else:
    if DS.exists():
        shutil.rmtree(DS)

    train_records = det_train + vid_train
    sel = sample_plan(plan_tiles(train_records, TILE, OVERLAP), MAX_TRAIN_TILES, EMPTY_TILE_FRAC, rng)
    write_tiles(train_records, sel, DS / "images/train", DS / "labels/train", copy_paste_prob=COPY_PASTE_PROB, rng=rng)

    # validation tiles (used only to pick the best epoch; the headline metric is the COCO evaluation on full frames below)
    val_sel = sample_plan(plan_tiles(det_val, TILE, OVERLAP), 1200, 0.10, rng)
    write_tiles(det_val, val_sel, DS / "images/val", DS / "labels/val", copy_paste_prob=0.0, rng=rng)

    DATA_YAML.write_text(f"path: {DS}\ntrain: images/train\nval: images/val\nnames:\n  0: person\n")
    print(DATA_YAML.read_text())
    del sel, val_sel, train_records

del det_train, vid_train          # only the validation records are needed from here on
gc.collect(); ram("after tiles")

DET train: 6471 images, 106396 person boxes, 83% COCO-small (<32px side), median side 17px
DET val: 548 images, 13969 person boxes, 89% COCO-small (<32px side), median side 17px
VID train (subsampled): 3050 images, 41594 person boxes, 39% COCO-small (<32px side), median side 36px
VID val (subsampled): 288 images, 5096 person boxes, 25% COCO-small (<32px side), median side 47px


planning tiles:   0%|          | 0/9521 [00:00<?, ?it/s]

writing tiles:   0%|          | 0/7212 [00:00<?, ?it/s]

wrote 16000 tiles, 94932 boxes (14317 pasted) -> /content/visdrone_work/tiles/images/train


planning tiles:   0%|          | 0/548 [00:00<?, ?it/s]

writing tiles:   0%|          | 0/497 [00:00<?, ?it/s]

wrote 1200 tiles, 11162 boxes (0 pasted) -> /content/visdrone_work/tiles/images/val
path: /content/visdrone_work/tiles
train: images/train
val: images/val
names:
  0: person

[RAM after tiles] system 1.6/12.7 GB | this notebook 0.6 GB + workers 0.0 GB


## 7. Blur augmentation + NWD loss (patches Ultralytics; self-tests run immediately)

In [ ]:
# =====================  (A) Motion-blur / degradation augmentation  =====================
# Hooks into Ultralytics' augmentation pipeline (runs after mosaic) so it works without albumentations
# and across Ultralytics 8.3.x versions. Pure image ops -> bounding boxes stay valid.
import ultralytics.data.augment as UA

P_MOTION, P_DEFOCUS, P_NOISE, P_JPEG, P_BC = 0.35, 0.12, 0.15, 0.20, 0.30

def _motion_kernel(k, angle):
    ker = np.zeros((k, k), np.float32); ker[k // 2, :] = 1.0
    M = cv2.getRotationMatrix2D(((k - 1) / 2, (k - 1) / 2), angle, 1.0)
    ker = cv2.warpAffine(ker, M, (k, k))
    s = ker.sum()
    return ker / s if s > 0 else np.pad(np.ones((1, 1), np.float32), k // 2)

def degrade(img, rnd=random):
    """Random drone-style degradation of a uint8 BGR image."""
    if rnd.random() < P_MOTION:                       # camera/platform motion blur
        k = rnd.choice([3, 5, 7, 9, 11, 13])
        img = cv2.filter2D(img, -1, _motion_kernel(k, rnd.uniform(0, 180)))
    elif rnd.random() < P_DEFOCUS:                    # defocus / Gaussian blur
        img = cv2.GaussianBlur(img, (0, 0), rnd.uniform(0.6, 1.8))
    if rnd.random() < P_BC:                           # exposure changes
        img = cv2.convertScaleAbs(img, alpha=rnd.uniform(0.75, 1.25), beta=rnd.uniform(-20, 20))
    if rnd.random() < P_NOISE:                        # sensor noise
        noise = np.random.normal(0, rnd.uniform(2, 7), img.shape).astype(np.float32)
        img = np.clip(img.astype(np.float32) + noise, 0, 255).astype(np.uint8)
    if rnd.random() < P_JPEG:                         # video-compression artefacts
        ok, enc = cv2.imencode(".jpg", img, [cv2.IMWRITE_JPEG_QUALITY, rnd.randint(40, 90)])
        if ok:
            img = cv2.imdecode(enc, cv2.IMREAD_COLOR)
    return img

_orig_alb_call = UA.Albumentations.__call__
def _alb_call(self, labels):
    img = labels.get("img")
    if img is not None and img.dtype == np.uint8 and img.ndim == 3 and img.shape[2] == 3:
        labels["img"] = degrade(np.ascontiguousarray(img))
    return _orig_alb_call(self, labels)
UA.Albumentations.__call__ = _alb_call

# =====================  (B) NWD + CIoU box loss (tiny-object friendly)  =====================
import torch
import ultralytics.utils.loss as UL

_ctx = {"stride": None, "fg": None}
_orig_make_anchors, _orig_bbox_iou, _orig_bbox_forward = UL.make_anchors, UL.bbox_iou, UL.BboxLoss.forward

def _make_anchors(*a, **k):
    out = _orig_make_anchors(*a, **k)
    _ctx["stride"] = out[1]                           # (num_anchors, 1) stride of every anchor
    return out

def _bbox_forward(self, *args, **kw):
    _ctx["fg"] = kw.get("fg_mask", args[6] if len(args) > 6 else None)
    try:
        return _orig_bbox_forward(self, *args, **kw)
    finally:
        _ctx["fg"] = None

def _bbox_iou(box1, box2, xywh=True, GIoU=False, DIoU=False, CIoU=False, eps=1e-7):
    iou = _orig_bbox_iou(box1, box2, xywh=xywh, GIoU=GIoU, DIoU=DIoU, CIoU=CIoU, eps=eps)
    st, fg = _ctx["stride"], _ctx["fg"]
    if not CIoU or xywh or st is None or fg is None or fg.ndim != 2 or st.numel() != fg.shape[1] or int(fg.sum()) != box1.shape[0]:
        return iou                                    # anything unexpected -> plain CIoU (never crashes training)
    s = st.view(1, -1).expand(fg.shape[0], -1)[fg].unsqueeze(-1).to(box1.dtype)   # per-box stride, grid units -> pixels
    b1, b2 = box1 * s, box2 * s
    (x1, y1, x2, y2), (u1, v1, u2, v2) = b1.chunk(4, -1), b2.chunk(4, -1)
    w2sq = (((x1 + x2) - (u1 + u2)) / 2) ** 2 + (((y1 + y2) - (v1 + v2)) / 2) ** 2 \
         + (((x2 - x1) - (u2 - u1)) / 2) ** 2 + (((y2 - y1) - (v2 - v1)) / 2) ** 2   # squared 2-Wasserstein distance of the two Gaussians
    nwd = torch.exp(-torch.sqrt(w2sq + eps) / NWD_C)
    return (1 - NWD_WEIGHT) * iou + NWD_WEIGHT * nwd.to(iou.dtype)

UL.make_anchors, UL.bbox_iou, UL.BboxLoss.forward = _make_anchors, _bbox_iou, _bbox_forward

# =====================  (C) Memory-safe box matching (TaskAlignedAssigner)  =====================
# Matching memory grows with  batch x (most persons in any tile) x (anchors; the P2 head adds ~37k).
# One crowded tile used to overflow the GPU, Ultralytics then retried on the CPU and used up all system RAM.
# Fix: when a batch is "big", match one image at a time, and only against that image's real boxes (padding removed).
import ultralytics.utils.tal as TAL
TAL_MAX_BOXES = 800      # batch x max-persons above which images are matched one by one
_orig_tal_forward = TAL.TaskAlignedAssigner.forward

@torch.no_grad()
def _tal_forward(self, *args, **kw):
    if kw or len(args) != 6:
        return _orig_tal_forward(self, *args, **kw)
    pd_scores, pd_bboxes, anc_points, gt_labels, gt_bboxes, mask_gt = args
    B, n = gt_bboxes.shape[:2]
    if n == 0 or B * n <= TAL_MAX_BOXES:
        return _orig_tal_forward(self, *args)
    outs = []
    for i in range(B):
        real = mask_gt[i].reshape(-1).nonzero()
        k = int(real.max()) + 1 if len(real) else 1       # padding sits at the end of each image's box list
        outs.append(_orig_tal_forward(self, pd_scores[i:i + 1], pd_bboxes[i:i + 1], anc_points,
                                      gt_labels[i:i + 1, :k], gt_bboxes[i:i + 1, :k], mask_gt[i:i + 1, :k]))
    return tuple(torch.cat(t, 0) for t in zip(*outs))

TAL.TaskAlignedAssigner.forward = _tal_forward

def _selftest_tal():
    """Chunked matching must give exactly the same assignment as the original; otherwise the patch is switched off."""
    global TAL_MAX_BOXES
    keep = TAL_MAX_BOXES
    try:
        torch.manual_seed(0)
        B, A, n = 3, 400, 7
        g = torch.arange(20).float()
        anc = torch.stack(torch.meshgrid(g, g, indexing="xy"), -1).reshape(-1, 2) + 0.5          # 400 anchors
        scores = torch.rand(B, A, 1)
        xy = torch.rand(B, n, 2) * 12; wh = torch.rand(B, n, 2) * 4 + 2
        gt = torch.cat([xy, xy + wh], -1)
        mask = torch.zeros(B, n, 1, dtype=torch.bool)
        for i, k in enumerate([7, 3, 0]):
            mask[i, :k] = True
        gt = gt * mask; labels = torch.zeros(B, n, 1)
        pd = gt[:, :1].repeat(1, A, 1) + torch.randn(B, A, 4)
        a = TAL.TaskAlignedAssigner(topk=10, num_classes=1)
        TAL_MAX_BOXES = 10 ** 9; ref = a(scores, pd, anc, labels, gt, mask)
        TAL_MAX_BOXES = 0;       got = a(scores, pd, anc, labels, gt, mask)
        assert len(ref) == len(got) and all(torch.allclose(r.float(), q.float(), atol=1e-5) for r, q in zip(ref, got)), "outputs differ"
        print("assigner patch self-test passed (chunked == original)")
    except Exception as e:
        TAL.TaskAlignedAssigner.forward = _orig_tal_forward
        print("!! assigner patch self-test FAILED, patch disabled:", repr(e)[:200])
    finally:
        TAL_MAX_BOXES = keep

# ---------- self-tests (run instantly; fail loudly if a patch is broken) ----------
def _selftest():
    b = torch.tensor([[10., 10., 14., 18.], [30., 30., 40., 50.]])
    _ctx["stride"] = torch.full((4, 1), 4.0); _ctx["fg"] = torch.tensor([[1, 0, 1, 0]], dtype=torch.bool)
    same = _bbox_iou(b, b.clone(), xywh=False, CIoU=True)
    shifted = _bbox_iou(b, b + torch.tensor([1., 0., 1., 0.]), xywh=False, CIoU=True)
    assert same.shape == (2, 1) and torch.allclose(same, torch.ones(2, 1), atol=1e-4), same
    assert (shifted < same).all(), shifted
    _ctx["stride"] = _ctx["fg"] = None
    im = (np.random.rand(256, 256, 3) * 255).astype(np.uint8)
    assert degrade(im.copy()).shape == im.shape
    print("self-tests passed: NWD loss shape/values OK, augmentation OK")
_selftest()
_selftest_tal()

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
self-tests passed: NWD loss shape/values OK, augmentation OK
assigner patch self-test passed (chunked == original)


## 8. Train YOLOv8-P2 (person, one class)

In [ ]:
from ultralytics import YOLO

# P2 (stride-4) head + pretrained COCO weights for every layer whose shape matches
model = YOLO(f"yolov8{MODEL_SIZE}-p2.yaml").load(f"yolov8{MODEL_SIZE}.pt")

train_args = dict(
    data=str(DATA_YAML), imgsz=IMGSZ, epochs=EPOCHS, batch=BATCH, workers=WORKERS,
    project=str(RUNS), name=f"person_p2_nwd_{MODEL_SIZE}", exist_ok=True,
    cos_lr=True, patience=15, amp=True, seed=SEED, max_det=300, plots=True,
    mosaic=1.0, close_mosaic=3,        # mosaic helps; last epochs without it to match test-time statistics
    scale=0.3,                         # mild scale jitter (default 0.5 would shrink already-tiny people too much)
    translate=0.1, fliplr=0.5, flipud=0.0, degrees=0.0, mixup=0.0, copy_paste=0.0,  # box-level copy-paste was done offline
)
_n = {"b": 0}
def _ram_cb(trainer):
    _n["b"] += 1
    if _n["b"] % 500 == 0:
        ram(f"epoch {trainer.epoch + 1}, batch {_n['b']}")
model.add_callback("on_train_batch_end", _ram_cb)
model.add_callback("on_train_epoch_end", lambda tr: (gc.collect(), ram(f"end of epoch {tr.epoch + 1}")))
ram("before training")
model.train(**train_args)
BEST = Path(model.trainer.best)
print("best weights:", BEST)

Transferred 219/437 items from pretrained weights
[RAM before training] system 1.8/12.7 GB | this notebook 0.8 GB + workers 0.0 GB
New https://pypi.org/project/ultralytics/8.4.190 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.253 🚀 Python-3.13.16 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=3, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/content/visdrone_work/tiles/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=30, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=768, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_

## 9. Sliced detector

In [ ]:
def nms_np(boxes, scores, thr):
    order = scores.argsort()[::-1]; keep = []
    area = (boxes[:, 2] - boxes[:, 0]) * (boxes[:, 3] - boxes[:, 1])
    while len(order):
        i = order[0]; keep.append(i)
        if len(order) == 1:
            break
        r = order[1:]
        w = (np.minimum(boxes[i, 2], boxes[r, 2]) - np.maximum(boxes[i, 0], boxes[r, 0])).clip(0)
        h = (np.minimum(boxes[i, 3], boxes[r, 3]) - np.maximum(boxes[i, 1], boxes[r, 1])).clip(0)
        inter = w * h
        order = r[inter / (area[i] + area[r] - inter + 1e-9) <= thr]
    return np.array(keep, dtype=int)

def nms(boxes, scores, thr):
    try:
        import torch, torchvision
        return torchvision.ops.nms(torch.from_numpy(boxes).float(), torch.from_numpy(scores).float(), thr).numpy()
    except ImportError:
        return nms_np(boxes, scores, thr)

class SlicedDetector:
    """Tiled (SAHI-style) person detector. Tiles use the SAME size/overlap as training.
    mode="sliced": tiles (+ optional full-frame pass for large people) -> merged with NMS.
    mode="full"  : a single full-frame pass at `full_imgsz` (baseline for comparison)."""
    def __init__(self, weights=None, model=None, tile=TILE, overlap=OVERLAP, imgsz=IMGSZ, conf=0.001,
                 tile_iou=0.6, merge_iou=0.6, mode="sliced", full_pass=True, full_imgsz=1280, full_min_side=28,
                 tile_batch=16, max_det=500, edge_margin=2):
        if model is None:
            from ultralytics import YOLO
            model = YOLO(str(weights))
        self.model, self.tile, self.overlap, self.imgsz, self.conf = model, tile, overlap, imgsz, conf
        self.tile_iou, self.merge_iou, self.mode = tile_iou, merge_iou, mode
        self.full_pass, self.full_imgsz, self.full_min_side = full_pass, full_imgsz, full_min_side
        self.tile_batch, self.max_det, self.edge_margin = tile_batch, max_det, edge_margin
        try:
            import torch; self.half = torch.cuda.is_available()
        except ImportError:
            self.half = False

    def _infer(self, imgs, imgsz):
        res = self.model.predict(imgs, imgsz=imgsz, conf=self.conf, iou=self.tile_iou, half=self.half,
                                 max_det=300, verbose=False)
        out = []
        for r in res:
            b = r.boxes
            out.append(np.concatenate([b.xyxy.cpu().numpy(), b.conf.cpu().numpy()[:, None]], 1).astype(np.float32)
                       if len(b) else np.zeros((0, 5), np.float32))
        return out

    def _finalize(self, d):
        if len(d) == 0:
            return np.zeros((0, 5), np.float32)
        d = d[nms(d[:, :4], d[:, 4], self.merge_iou)]
        return d[np.argsort(-d[:, 4])][: self.max_det]

    def __call__(self, img):
        H, W = img.shape[:2]
        if self.mode == "full":
            return self._finalize(self._infer([img], self.full_imgsz)[0])
        dets, origins, m = [], tile_origins(W, H, self.tile, self.overlap), self.edge_margin
        for i in range(0, len(origins), self.tile_batch):
            chunk = origins[i:i + self.tile_batch]
            crops = [img[y:y + self.tile, x:x + self.tile] for x, y in chunk]
            for (x, y), crop, d in zip(chunk, crops, self._infer(crops, self.imgsz)):
                if len(d) == 0:
                    continue
                th, tw = crop.shape[:2]
                keep = np.ones(len(d), bool)           # drop boxes cut by an INTERIOR tile border (a neighbour tile sees them whole)
                if x > 0:       keep &= d[:, 0] > m
                if y > 0:       keep &= d[:, 1] > m
                if x + tw < W:  keep &= d[:, 2] < tw - m
                if y + th < H:  keep &= d[:, 3] < th - m
                d = d[keep].copy()
                d[:, [0, 2]] += x; d[:, [1, 3]] += y
                dets.append(d)
        if self.full_pass:                              # large people that no single tile contains
            d = self._infer([img], self.full_imgsz)[0]
            if len(d):
                dets.append(d[np.sqrt((d[:, 2] - d[:, 0]) * (d[:, 3] - d[:, 1])) >= self.full_min_side])
        return self._finalize(np.concatenate(dets) if dets else np.zeros((0, 5), np.float32))

## 10. COCO evaluation helpers

In [ ]:
def blur_score(img):
    """Variance of the Laplacian: LOW = blurry. Computed at a fixed width so scores are comparable across frames."""
    g = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    s = 960.0 / max(g.shape)
    if s < 1:
        g = cv2.resize(g, None, fx=s, fy=s, interpolation=cv2.INTER_AREA)
    return float(cv2.Laplacian(g, cv2.CV_64F).var())

def in_ignore(dets, ign, thr=0.5):
    """VisDrone protocol: a detection mostly (>=50% of its area) inside an ignored region is neither TP nor FP."""
    if len(ign) == 0 or len(dets) == 0:
        return np.zeros(len(dets), bool)
    d, g = dets[:, None, :4], ign[None, :, :]
    wh = (np.minimum(d[..., 2:], g[..., 2:]) - np.maximum(d[..., :2], g[..., :2])).clip(0)
    inter = wh[..., 0] * wh[..., 1]
    area = ((dets[:, 2] - dets[:, 0]) * (dets[:, 3] - dets[:, 1]))[:, None]
    return ((inter / np.maximum(area, 1e-9)) >= thr).any(1)

def run_detection(detector, records, max_images=None, desc="detect"):
    recs = records[:max_images] if max_images else records
    preds, blur, t0 = [], [], time.time()
    for i, r in enumerate(tqdm(recs, desc=desc)):
        img = cv2.imread(r["path"])
        blur.append(blur_score(img))
        d = detector(img)
        d = d[~in_ignore(d, r["ignore"])]
        for x1, y1, x2, y2, c in d:
            preds.append(dict(image_id=i + 1, category_id=1, bbox=[float(x1), float(y1), float(x2 - x1), float(y2 - y1)], score=float(c)))
    return recs, preds, np.array(blur), (time.time() - t0) / max(len(recs), 1)

def build_gt(records):
    images, anns, aid = [], [], 1
    for i, r in enumerate(records):
        W, H = Image.open(r["path"]).size
        images.append(dict(id=i + 1, width=W, height=H, file_name=Path(r["path"]).name))
        for x1, y1, x2, y2 in r["boxes"]:
            w, h = float(x2 - x1), float(y2 - y1)
            anns.append(dict(id=aid, image_id=i + 1, category_id=1, bbox=[float(x1), float(y1), w, h], area=w * h, iscrowd=0)); aid += 1
    return dict(images=images, annotations=anns, categories=[dict(id=1, name="person")])

def coco_eval(records, preds, img_ids=None, max_dets=100):
    """Official COCO bbox evaluation (IoU .50:.95). small = area < 32^2 px, medium < 96^2, large >= 96^2."""
    from pycocotools.coco import COCO
    from pycocotools.cocoeval import COCOeval
    keys = ["AP", "AP50", "AP75", "AP_small", "AP_medium", "AP_large"]
    gt = COCO()
    gt.dataset = build_gt(records)
    with contextlib.redirect_stdout(io.StringIO()):
        gt.createIndex()
    if img_ids is not None:
        ids = set(int(i) for i in img_ids); preds = [p for p in preds if p["image_id"] in ids]
    if not preds:
        return {k: 0.0 for k in keys}
    with contextlib.redirect_stdout(io.StringIO()):
        dt = gt.loadRes(preds)
        E = COCOeval(gt, dt, "bbox")
        E.params.maxDets = [1, 10, max_dets]
        if img_ids is not None:
            E.params.imgIds = sorted(ids)
        E.evaluate(); E.accumulate(); E.summarize()
    return {k: float(v) for k, v in zip(keys, E.stats[:6])}

def report(records, preds, blur, title):
    """Headline table + sharp-vs-blurry breakdown (blurriest 25% of frames vs the rest)."""
    rows = {}
    rows["all frames  (maxDets=100, standard COCO)"] = coco_eval(records, preds)
    rows["all frames  (maxDets=500, crowded scenes)"] = coco_eval(records, preds, max_dets=500)
    ids = np.arange(1, len(records) + 1); thr = np.percentile(blur, 25)
    rows["blurriest 25% of frames"] = coco_eval(records, preds, ids[blur <= thr], 500)
    rows["sharper 75% of frames"]   = coco_eval(records, preds, ids[blur > thr], 500)
    print(f"\n=== {title} ===  ({len(records)} images)")
    print(f"{'':44s}" + "".join(f"{k:>10s}" for k in ["AP", "AP50", "AP_small", "AP_med", "AP_large"]))
    for name, m in rows.items():
        print(f"{name:44s}" + "".join(f"{m[k]*100:10.1f}" for k in ["AP", "AP50", "AP_small", "AP_medium", "AP_large"]))
    return rows

## 11. Evaluate: sliced vs full-frame, DET and VID, blurry vs sharp frames
The `AP_small` column is the metric you asked for (COCO definition: IoU 0.50:0.95, objects with area < 32x32 px).

In [ ]:
BEST_W = BEST if "BEST" in globals() else next(iter(sorted(RUNS.glob("person_p2_nwd_*/weights/best.pt"))))
det = SlicedDetector(BEST_W)

# --- image-detection val set (DET) ---
recs, preds, blur, sec = run_detection(det, det_val, EVAL_MAX_IMAGES, "DET val | sliced")
res_sliced = report(recs, preds, blur, "VisDrone-DET val | sliced inference")
print(f"~{sec:.2f} s / frame")

# --- ablation: plain full-frame inference at 1280 (shows what slicing buys) ---
gc.collect(); ram("after sliced eval")
det.mode = "full"
recs2, preds2, blur2, sec2 = run_detection(det, det_val, EVAL_MAX_IMAGES, "DET val | full-frame 1280")
res_full = report(recs2, preds2, blur2, "VisDrone-DET val | full-frame @1280 (baseline)")
det.mode = "sliced"

# --- video val set (VID frames) : this is the number that matters for video detection ---
if vid_val:
    recs3, preds3, blur3, sec3 = run_detection(det, vid_val, EVAL_MAX_IMAGES, "VID val | sliced")
    res_vid = report(recs3, preds3, blur3, "VisDrone-VID val frames | sliced inference")
    BLUR_THR = float(np.percentile(blur3, 25))
else:
    BLUR_THR = float(np.percentile(blur, 25))
print("blur threshold (25th percentile of val frames):", round(BLUR_THR, 1))

# --- save everything to Google Drive ---
ALL = {"DET_sliced": res_sliced, "DET_fullframe_1280": res_full}
if vid_val:
    ALL["VID_sliced"] = res_vid
(RESULTS / "metrics.json").write_text(json.dumps(ALL, indent=1))
with open(RESULTS / "metrics.csv", "w") as f:
    f.write("dataset,subset,AP,AP50,AP75,AP_small,AP_medium,AP_large\n")
    for ds, rows in ALL.items():
        for sub, m in rows.items():
            f.write(f"{ds},{sub.strip()}," + ",".join(f"{m[k]:.4f}" for k in ["AP", "AP50", "AP75", "AP_small", "AP_medium", "AP_large"]) + "\n")
(RESULTS / "predictions_DET_val_sliced.json").write_text(json.dumps(preds))
shutil.copy(BEST_W, WEIGHTS / "best.pt")
print("saved metrics.json / metrics.csv / predictions / best.pt to", OUT)

## 12. Video inference (detection only)

In [ ]:
CONF_SHARP, CONF_BLUR = 0.30, 0.20   # blurred frames produce weaker scores -> use a lower confidence cut-off there

def process_video(src, dst, detector, max_frames=None, blur_thr=None, save_json=None):
    """Per-frame person detection on a video file (no tracking). Returns the list of per-frame detections."""
    blur_thr = BLUR_THR if blur_thr is None else blur_thr
    cap = cv2.VideoCapture(str(src))
    fps = cap.get(cv2.CAP_PROP_FPS) or 25
    W, H = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    n = int(cap.get(cv2.CAP_PROP_FRAME_COUNT)) or None
    raw = str(WORK / "_raw_video.mp4")   # write locally, then encode straight into Drive
    out = cv2.VideoWriter(raw, cv2.VideoWriter_fourcc(*"mp4v"), fps, (W, H))
    log, t0, k = [], time.time(), 0
    pbar = tqdm(total=min(n, max_frames) if (n and max_frames) else (max_frames or n), desc="video")
    while True:
        ok, frame = cap.read()
        if not ok or (max_frames and k >= max_frames):
            break
        blurry = blur_score(frame) < blur_thr
        d = detector(frame); d = d[d[:, 4] >= (CONF_BLUR if blurry else CONF_SHARP)]
        for x1, y1, x2, y2, c in d:
            cv2.rectangle(frame, (int(x1), int(y1)), (int(x2), int(y2)), (0, 255, 255) if blurry else (0, 255, 0), 1)
        cv2.putText(frame, f"persons: {len(d)}  {'BLURRY' if blurry else 'sharp'}", (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 255, 255), 2)
        out.write(frame)
        log.append(dict(frame=k, blurry=bool(blurry), boxes=[[float(v) for v in row] for row in d]))
        k += 1; pbar.update(1)
    cap.release(); out.release(); pbar.close()
    subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", raw, "-vcodec", "libx264", "-pix_fmt", "yuv420p", str(dst)], check=False)
    if save_json:
        Path(save_json).write_text(json.dumps(log))
    print(f"{k} frames in {time.time()-t0:.0f}s ({(time.time()-t0)/max(k,1):.2f} s/frame) -> {dst}")
    return log

# ---- demo: turn the first VID-val sequence into a short clip, run it, and show it ----
if VID_VAL:
    seq = sorted(d for d in (Path(VID_VAL) / "sequences").iterdir() if d.is_dir())[0]
    frames = sorted(seq.glob("*.jpg"))[:150]
    h, w = cv2.imread(str(frames[0])).shape[:2]
    demo_in = WORK / "demo_in.mp4"
    vw = cv2.VideoWriter(str(demo_in), cv2.VideoWriter_fourcc(*"mp4v"), 25, (w, h))
    for f in frames: vw.write(cv2.imread(str(f)))
    vw.release()
    process_video(demo_in, RESULTS / "demo_out.mp4", det, save_json=RESULTS / "demo_detections.json")
    from IPython.display import Video, display
    display(Video(str(RESULTS / "demo_out.mp4"), embed=True, width=800))

# ---- your own video: upload it in the Colab file panel (or point to a file in your Drive), then run e.g. ----
# process_video("/content/my_drone_clip.mp4", RESULTS / "my_drone_clip_out.mp4", det, save_json=RESULTS / "my_drone_clip.json")

## 13. Resume after a disconnect (re-run sections 1-5 and 7 first)

In [ ]:
from ultralytics import YOLO
last = sorted(RUNS.glob("person_p2_nwd_*/weights/last.pt"))[-1]
YOLO(str(last)).train(resume=True)